# 00 — Native data exploration and statistical assessment

This notebook describes the native data before translation or modelling.

It produces:

- exact dataset and file inventories;
- a complete feature dictionary;
- visible examples of native observations;
- missing, frozen and duplicate-value assessments;
- descriptive distributions and dependence summaries;
- complete selected time-series plots;
- cadence, gaps, autocorrelation, trend, spectrum and stationarity statistics;
- label and period summaries;
- for Petrobras 3W, an eligibility table matching the anomaly-detection benchmark
  in Vargas et al. (2019).

No detector is fitted and no product or operational interpretation is made here.


## 1. Setup

The default sector is telecom. Change `EDA_SECTOR` to `petrobras_3w` before running
the notebook to explore 3W.


In [ ]:
import configparser
import hashlib
import json
import math
import os
import re
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import scipy
from scipy import signal, stats
from statsmodels.tsa.stattools import adfuller, kpss
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import discover_telecom, discover_threew, native_path

EDA_SECTOR = os.getenv("EDA_SECTOR", "telecom")
assert EDA_SECTOR in {"telecom", "petrobras_3w"}

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EDA_RUN_ID = os.getenv("EDA_RUN_ID", f"{EDA_SECTOR}_eda_{RUN_STAMP}")
RANDOM_SEED = int(os.getenv("EDA_RANDOM_SEED", "42"))
SAMPLE_ROWS = int(os.getenv("EDA_SAMPLE_ROWS", "200000"))
THREEW_SAMPLE_FILES = int(os.getenv("EDA_THREEW_SAMPLE_FILES", "30"))
THREEW_ROWS_PER_FILE = int(os.getenv("EDA_THREEW_ROWS_PER_FILE", "5000"))
SERIES_COUNT = int(os.getenv("EDA_SERIES_COUNT", "4"))
SERIES_MAX_PLOT_POINTS = int(os.getenv("EDA_SERIES_MAX_PLOT_POINTS", "4000"))
FULL_LABEL_SCAN = os.getenv("EDA_FULL_LABEL_SCAN", "1") == "1"

TELECOM_SOURCE = Path(os.getenv(
    "TELECOM_SOURCE_ROOT",
    str(
        DRIVE_ROOT / "Full dataset"
        if (DRIVE_ROOT / "Full dataset").exists()
        else DRIVE_ROOT
    ),
))
THREEW_SOURCE = Path(os.getenv(
    "THREEW_SOURCE_ROOT",
    str(
        DRIVE_ROOT / "sources" / "petrobras_3w" / "2.0.0"
        / "raw" / "3w_dataset_2.0.0"
    ),
))
SOURCE_ROOT = TELECOM_SOURCE if EDA_SECTOR == "telecom" else THREEW_SOURCE
OUTPUT = DRIVE_ROOT / "outputs" / "exploration" / EDA_SECTOR / EDA_RUN_ID
FIGURES = OUTPUT / "figures"
if OUTPUT.exists():
    raise FileExistsError(f"Output already exists: {OUTPUT}. Choose a new EDA_RUN_ID.")
FIGURES.mkdir(parents=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)
rng = np.random.default_rng(RANDOM_SEED)

display(pd.Series({
    "sector": EDA_SECTOR,
    "source": str(SOURCE_ROOT),
    "output": str(OUTPUT),
    "distribution_sample_rows": SAMPLE_ROWS,
    "selected_complete_series": SERIES_COUNT,
    "random_seed": RANDOM_SEED,
}, name="value").to_frame())


In [ ]:
def hash_fraction(value, namespace="sample"):
    raw = f"{RANDOM_SEED}|{namespace}|{value}".encode("utf-8")
    return int(hashlib.sha256(raw).hexdigest()[:16], 16) / 16**16


def opaque_id(value, prefix="instance"):
    raw = f"{RANDOM_SEED}|{value}".encode("utf-8")
    return f"{prefix}_{hashlib.sha256(raw).hexdigest()[:16]}"


def threew_source_kind(path):
    if path.stem.startswith("WELL-"):
        return "real"
    if path.stem.startswith("SIMULATED_"):
        return "simulated"
    if path.stem.startswith("DRAWN_"):
        return "hand_drawn"
    return "other"


def read_threew(path, columns):
    frame = pd.read_parquet(path, columns=columns)
    if "timestamp" not in frame.columns:
        frame = frame.reset_index()
    return frame


def timestamp_bounds(parquet, field):
    if field not in parquet.schema.names:
        return pd.NaT, pd.NaT
    column_index = parquet.schema.names.index(field)
    starts, ends = [], []
    for row_group in range(parquet.num_row_groups):
        item = parquet.metadata.row_group(row_group).column(column_index)
        statistics = item.statistics
        if statistics is not None and statistics.has_min_max:
            starts.append(pd.to_datetime(statistics.min, utc=True))
            ends.append(pd.to_datetime(statistics.max, utc=True))
    return (
        min(starts) if starts else pd.NaT,
        max(ends) if ends else pd.NaT,
    )


def metadata_column_profile(parquet, field):
    column_index = parquet.schema.names.index(field)
    total_rows = parquet.metadata.num_rows
    null_count = 0
    minima, maxima = [], []
    statistics_complete = True
    for row_group in range(parquet.num_row_groups):
        item = parquet.metadata.row_group(row_group).column(column_index)
        statistics = item.statistics
        group_rows = parquet.metadata.row_group(row_group).num_rows
        if statistics is None:
            statistics_complete = False
            continue
        null_count += int(statistics.null_count or 0)
        observed = group_rows - int(statistics.null_count or 0)
        if observed:
            if statistics.has_min_max:
                minima.append(statistics.min)
                maxima.append(statistics.max)
            else:
                statistics_complete = False
    all_missing = null_count == total_rows
    frozen = (
        null_count == 0
        and statistics_complete
        and len(minima) > 0
        and min(minima) == max(maxima)
    )
    return {
        "rows": total_rows,
        "null_count": null_count,
        "missing_fraction": null_count / total_rows if total_rows else np.nan,
        "all_missing": all_missing,
        "frozen_non_missing_values": frozen,
        "minimum": min(minima) if minima else np.nan,
        "maximum": max(maxima) if maxima else np.nan,
        "metadata_statistics_complete": statistics_complete,
    }


def write_csv(frame, filename, index=False):
    path = OUTPUT / filename
    frame.to_csv(path, index=index)
    return path


def write_json(payload, filename):
    path = OUTPUT / filename
    path.write_text(
        json.dumps(payload, indent=2, sort_keys=True, default=str) + "\n",
        encoding="utf-8",
    )
    return path


def save_figure(figure, filename):
    figure.tight_layout()
    figure.savefig(FIGURES / filename, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(figure)


## 2. Feature dictionary


In [ ]:
TELECOM_METRIC_INFO = {
    "rx_power_dbm": ("Received optical power", "dBm", "continuous gauge"),
    "tx_power_dbm": ("Transmitted optical power", "dBm", "continuous gauge"),
    "temperature_c": ("ONT temperature", "degC", "continuous gauge"),
    "bias_current_ma": ("Laser bias current", "mA", "continuous gauge"),
    "voltage_v": ("ONT supply voltage", "V", "continuous gauge"),
    "ber": ("Bit error ratio", "ratio", "bounded fraction"),
    "fec_count": ("Generator FEC-related interval count", "count", "interval count"),
    "crc_errors": ("CRC errors in the observation interval", "count", "interval count"),
    "uptime_s": ("Seconds since restart", "s", "cumulative counter"),
    "reboot_count": ("Cumulative reboot count", "count", "cumulative counter"),
    "throughput_mbps": ("Service throughput", "Mbps", "continuous gauge"),
}
TELECOM_CONTEXT = {
    "timestamp_utc": ("Observation timestamp", "timestamp"),
    "ont_id": ("Optical network terminal identifier", "identifier"),
    "olt_id": ("Optical line terminal identifier", "topology"),
    "pon_port": ("Passive optical network port", "topology"),
    "splitter_l1": ("First-level splitter identifier", "topology"),
    "splitter_l2": ("Second-level splitter identifier", "topology"),
    "geo_cluster": ("Geographic cluster", "context"),
    "device_model": ("ONT model", "context"),
    "vendor": ("ONT vendor", "context"),
    "enclosure": ("Installation enclosure", "context"),
    "firmware_version": ("Firmware version", "context"),
    "distance_m": ("Optical path distance", "context"),
    "distance_bucket": ("Distance category", "context"),
    "splitter_ratio": ("Optical splitter ratio", "context"),
    "l2_splitter_capacity": ("Second-level splitter capacity", "context"),
    "fibre_age_yr": ("Fibre age", "context"),
    "expected_rx_power_dbm": ("Expected received optical power", "context"),
    "rx_sensitivity_dbm": ("Receiver sensitivity", "context"),
    "service_impact_weight": ("Synthetic service impact weight", "context"),
    "customer_priority_weight": ("Synthetic customer priority weight", "context"),
}

if EDA_SECTOR == "telecom":
    discovery = discover_telecom(SOURCE_ROOT)
    assert discovery["core_ready"], discovery
    panel_path = native_path(SOURCE_ROOT, "reference_dataset.parquet")
    panel_parquet = pq.ParquetFile(panel_path)
    native_schema = pd.DataFrame([
        {"native_field": field.name, "physical_type": str(field.type)}
        for field in panel_parquet.schema_arrow
    ])
    feature_rows = []
    for record in native_schema.itertuples(index=False):
        field = record.native_field
        if field in TELECOM_METRIC_INFO:
            meaning, unit, kind = TELECOM_METRIC_INFO[field]
            role = "measurement"
        elif field.startswith("gt_"):
            meaning, unit, kind = (
                "Synthetic evaluation field",
                None,
                "label",
            )
            role = "evaluation"
        else:
            meaning, kind = TELECOM_CONTEXT.get(
                field, (field.replace("_", " "), "context")
            )
            unit = (
                "m" if field == "distance_m"
                else "yr" if field == "fibre_age_yr"
                else None
            )
            role = "identifier_or_context"
        feature_rows.append({
            "native_field": field,
            "meaning": meaning,
            "unit": unit,
            "physical_type": record.physical_type,
            "role": role,
            "measurement_kind": kind,
        })
    feature_dictionary = pd.DataFrame(feature_rows)
    timestamp_field, entity_field = "timestamp_utc", "ont_id"
    metric_fields = list(TELECOM_METRIC_INFO)
    context_fields = [
        field for field in TELECOM_CONTEXT
        if field not in {timestamp_field, entity_field}
        and field in native_schema["native_field"].tolist()
    ]
    label_fields = [
        field for field in native_schema["native_field"]
        if field.startswith("gt_")
    ]
else:
    discovery = discover_threew(SOURCE_ROOT)
    assert discovery["ready"], discovery
    parser = configparser.ConfigParser()
    parser.optionxform = str
    parser.read(SOURCE_ROOT / "dataset.ini")
    descriptions = dict(parser["PARQUET_FILE_PROPERTIES"])
    example_path = next(
        path
        for directory in range(10)
        for path in sorted((SOURCE_ROOT / str(directory)).glob("*.parquet"))
    )
    example_parquet = pq.ParquetFile(example_path)
    native_schema = pd.DataFrame([
        {"native_field": field.name, "physical_type": str(field.type)}
        for field in example_parquet.schema_arrow
    ])
    feature_rows = []
    for record in native_schema.itertuples(index=False):
        field = record.native_field
        description = descriptions.get(field, field)
        unit_match = re.search(r"\[([^\]]+)\]\s*$", description)
        unit = unit_match.group(1) if unit_match else None
        meaning = re.sub(r"\s*\[[^\]]+\]\s*$", "", description)
        if field == "timestamp":
            role, kind = "timestamp", "timestamp"
        elif field in {"class", "state"}:
            role, kind = "label", "categorical label"
        elif field.startswith("ESTADO-"):
            role, kind = "measurement", "discrete state"
        elif field.startswith("ABER-"):
            role, kind = "measurement", "bounded gauge"
        else:
            role, kind = "measurement", "continuous gauge"
        feature_rows.append({
            "native_field": field,
            "meaning": meaning,
            "unit": unit,
            "physical_type": record.physical_type,
            "role": role,
            "measurement_kind": kind,
        })
    feature_dictionary = pd.DataFrame(feature_rows)
    timestamp_field, entity_field = "timestamp", "entity_id"
    metric_fields = feature_dictionary.loc[
        feature_dictionary["role"].eq("measurement"), "native_field"
    ].tolist()
    context_fields = []
    label_fields = ["class", "state"]
    panel_path = None
    panel_parquet = None

write_csv(feature_dictionary, "feature_dictionary.csv")
display(feature_dictionary)
display(pd.Series({
    "measurement_count": len(metric_fields),
    "context_count": len(context_fields),
    "label_fields": ", ".join(label_fields),
}, name="count").to_frame())


For 3W 2.0.0, `dataset.ini` is used as the source for variable descriptions and
units, consistent with the current dataset documentation.


## 3. Exact inventory and structural dimensions


In [ ]:
if EDA_SECTOR == "telecom":
    topology = pd.read_csv(native_path(SOURCE_ROOT, "topology.csv"))
    service_windows = pd.read_csv(
        native_path(SOURCE_ROOT, "entity_service_windows.csv")
    )
    exact_start, exact_end = timestamp_bounds(panel_parquet, timestamp_field)
    files = [
        ("reference_dataset.parquet", "telemetry", False),
        ("topology.csv", "topology", False),
        ("entity_service_windows.csv", "service validity", False),
        ("engineering_events.csv", "operational events", False),
        ("tickets.csv", "evaluation", True),
        ("gt_fault_registry.csv", "evaluation", True),
        ("fault_entity_intervals.csv", "evaluation", True),
        ("gt_fault_groups.csv", "evaluation", True),
    ]
    inventory_rows = []
    for filename, role, evaluation in files:
        path = native_path(
            SOURCE_ROOT, filename,
            evaluation=evaluation, required=False,
        )
        if path is None:
            inventory_rows.append({
                "file": filename, "role": role, "present": False
            })
            continue
        if path.suffix == ".parquet":
            parquet = pq.ParquetFile(path)
            rows, columns = parquet.metadata.num_rows, len(parquet.schema.names)
        else:
            columns = len(pd.read_csv(path, nrows=0).columns)
            with path.open("r", encoding="utf-8", errors="replace") as handle:
                rows = max(0, sum(1 for _ in handle) - 1)
        inventory_rows.append({
            "file": filename,
            "role": role,
            "present": True,
            "rows": rows,
            "columns": columns,
            "size_mb": path.stat().st_size / 1024**2,
        })
    source_inventory = pd.DataFrame(inventory_rows)
    dataset_dimensions = pd.DataFrame([
        {"dimension": "telemetry rows", "value": panel_parquet.metadata.num_rows},
        {"dimension": "native fields", "value": len(panel_parquet.schema.names)},
        {"dimension": "ONTs", "value": topology["ont_id"].astype(str).nunique()},
        {"dimension": "OLTs", "value": topology["olt_id"].astype(str).nunique()},
        {"dimension": "PON ports", "value": topology["pon_port"].astype(str).nunique()},
        {"dimension": "L2 splitters", "value": topology["splitter_l2"].astype(str).nunique()},
        {"dimension": "time start", "value": exact_start},
        {"dimension": "time end", "value": exact_end},
    ])
    file_inventory = pd.DataFrame()
else:
    paths = sorted(
        path
        for event_code in range(10)
        for path in (SOURCE_ROOT / str(event_code)).glob("*.parquet")
    )
    inventory_rows = []
    metadata_quality_rows = []
    for position, path in enumerate(paths, start=1):
        parquet = pq.ParquetFile(path)
        start, end = timestamp_bounds(parquet, timestamp_field)
        source_kind = threew_source_kind(path)
        entity_id = (
            path.stem.split("_", 1)[0]
            if source_kind == "real" else path.stem
        )
        event_code = int(path.parent.name)
        instance_id = opaque_id(str(path.relative_to(SOURCE_ROOT)))
        inventory_rows.append({
            "instance_id": instance_id,
            "event_code": event_code,
            "source_kind": source_kind,
            "entity_id": entity_id,
            "rows": parquet.metadata.num_rows,
            "columns": len(parquet.schema.names),
            "start_ts": start,
            "end_ts": end,
            "duration_hours": (
                (end - start).total_seconds() / 3600
                if pd.notna(start) and pd.notna(end) else np.nan
            ),
            "size_mb": path.stat().st_size / 1024**2,
            "path": path,
        })
        for field in metric_fields:
            profile = metadata_column_profile(parquet, field)
            metadata_quality_rows.append({
                "instance_id": instance_id,
                "event_code": event_code,
                "source_kind": source_kind,
                "field": field,
                **profile,
            })
        if position % 400 == 0:
            print(f"Scanned {position:,}/{len(paths):,} Parquet metadata records")
    file_inventory = pd.DataFrame(inventory_rows)
    metadata_variable_quality = pd.DataFrame(metadata_quality_rows)
    source_inventory = (
        file_inventory.groupby(["source_kind", "event_code"], as_index=False)
        .agg(
            files=("instance_id", "size"),
            rows=("rows", "sum"),
            entities=("entity_id", "nunique"),
            duration_hours=("duration_hours", "sum"),
            size_mb=("size_mb", "sum"),
        )
    )
    dataset_dimensions = pd.DataFrame([
        {"dimension": "instances", "value": len(file_inventory)},
        {"dimension": "observations", "value": int(file_inventory["rows"].sum())},
        {"dimension": "measurement variables", "value": len(metric_fields)},
        {"dimension": "real wells", "value": file_inventory.loc[
            file_inventory["source_kind"].eq("real"), "entity_id"
        ].nunique()},
        {"dimension": "minimum instance rows", "value": int(file_inventory["rows"].min())},
        {"dimension": "median instance rows", "value": float(file_inventory["rows"].median())},
        {"dimension": "maximum instance rows", "value": int(file_inventory["rows"].max())},
        {"dimension": "time start", "value": file_inventory["start_ts"].min()},
        {"dimension": "time end", "value": file_inventory["end_ts"].max()},
    ])

write_csv(source_inventory, "source_inventory.csv")
write_csv(dataset_dimensions, "dataset_dimensions.csv")
if len(file_inventory):
    write_csv(
        file_inventory.drop(columns="path"),
        "threew_instance_inventory.csv",
    )
display(source_inventory)
display(dataset_dimensions)


## 4. Reproducible bounded sample and visible native records


In [ ]:
if EDA_SECTOR == "telecom":
    sample_columns = list(dict.fromkeys([
        timestamp_field, entity_field,
        *metric_fields, *context_fields,
        *[field for field in ["gt_state", "gt_fault_type"] if field in label_fields],
    ]))
    row_group_rows = np.array([
        panel_parquet.metadata.row_group(index).num_rows
        for index in range(panel_parquet.num_row_groups)
    ])
    raw_allocations = SAMPLE_ROWS * row_group_rows / row_group_rows.sum()
    allocations = np.floor(raw_allocations).astype(int)
    remaining = min(SAMPLE_ROWS, int(row_group_rows.sum())) - allocations.sum()
    for index in np.argsort(-(raw_allocations - allocations))[:remaining]:
        allocations[index] += 1
    sample_parts, manifest_rows = [], []
    for row_group, requested in enumerate(allocations):
        if requested == 0:
            continue
        frame = panel_parquet.read_row_group(
            row_group, columns=sample_columns
        ).to_pandas()
        take = min(len(frame), requested)
        selected = frame.sample(
            take, random_state=RANDOM_SEED + row_group
        )
        sample_parts.append(selected)
        manifest_rows.append({
            "source_unit": f"row_group_{row_group}",
            "source_rows": len(frame),
            "sample_rows": take,
        })
    analysis_sample = pd.concat(sample_parts, ignore_index=True)
    sample_manifest = pd.DataFrame(manifest_rows)
    series_key = entity_field
    sample_scope = "proportional random sample across all Parquet row groups"
else:
    real_inventory = file_inventory.loc[
        file_inventory["source_kind"].eq("real")
    ].copy()
    real_inventory["sample_order"] = real_inventory["instance_id"].map(
        lambda value: hash_fraction(value, "distribution")
    )
    per_event = max(1, math.ceil(THREEW_SAMPLE_FILES / 10))
    candidates = (
        real_inventory.sort_values(["event_code", "sample_order"])
        .groupby("event_code", sort=True)
        .head(per_event)
        .sort_values("sample_order")
    )
    selected_files = candidates.head(
        min(THREEW_SAMPLE_FILES, len(candidates))
    ).sort_values(["event_code", "sample_order"])
    sample_parts, manifest_rows = [], []
    for position, record in enumerate(selected_files.itertuples(index=False)):
        frame = read_threew(
            record.path,
            [timestamp_field, *metric_fields, *label_fields],
        )
        take = min(len(frame), THREEW_ROWS_PER_FILE)
        selected = frame.sample(
            take, random_state=RANDOM_SEED + position
        ).copy()
        selected[entity_field] = record.entity_id
        selected["instance_id"] = record.instance_id
        selected["event_code"] = record.event_code
        selected["source_kind"] = record.source_kind
        sample_parts.append(selected)
        manifest_rows.append({
            "instance_id": record.instance_id,
            "event_code": record.event_code,
            "source_kind": record.source_kind,
            "entity_id": record.entity_id,
            "source_rows": len(frame),
            "sample_rows": take,
        })
    analysis_sample = pd.concat(sample_parts, ignore_index=True)
    sample_manifest = pd.DataFrame(manifest_rows)
    series_key = "instance_id"
    sample_scope = "stratified sample of real instances across event directories"

analysis_sample[timestamp_field] = pd.to_datetime(
    analysis_sample[timestamp_field], utc=True, errors="coerce"
)
write_csv(sample_manifest, "sample_manifest.csv")
display(pd.Series({
    "sample_scope": sample_scope,
    "sample_rows": len(analysis_sample),
    "sample_entities": analysis_sample[entity_field].nunique(),
    "sample_time_start": analysis_sample[timestamp_field].min(),
    "sample_time_end": analysis_sample[timestamp_field].max(),
}, name="value").to_frame())
display(sample_manifest)


In [ ]:
default_focus = (
    ["rx_power_dbm", "temperature_c", "ber", "fec_count", "crc_errors", "throughput_mbps"]
    if EDA_SECTOR == "telecom"
    else ["P-ANULAR", "P-MON-CKP", "P-TPT", "QGL", "T-TPT", "ABER-CKP"]
)
requested_focus = [
    value.strip()
    for value in os.getenv("EDA_FOCUS_FEATURES", "").split(",")
    if value.strip()
]
focus_metrics = [
    field for field in (requested_focus or default_focus)
    if field in metric_fields
]
preview_columns = list(dict.fromkeys([
    timestamp_field, entity_field, *focus_metrics,
    *context_fields[:4],
    *[field for field in label_fields if field in analysis_sample],
    *[field for field in ["instance_id", "event_code", "source_kind"]
      if field in analysis_sample],
]))

print("First 10 sampled native observations:")
display(
    analysis_sample[preview_columns]
    .sort_values([series_key, timestamp_field])
    .head(10)
)
print("10 deterministic random observations:")
display(
    analysis_sample[preview_columns]
    .sample(min(10, len(analysis_sample)), random_state=RANDOM_SEED)
    .sort_values(timestamp_field)
)
print("One observation with fields shown vertically:")
display(analysis_sample[preview_columns].iloc[[0]].T.rename(columns={0: "value"}))
print("Definitions of the displayed measurement fields:")
display(
    feature_dictionary.loc[
        feature_dictionary["native_field"].isin(focus_metrics)
    ]
)


## 5. Missing, completely missing and frozen variables


In [ ]:
if EDA_SECTOR == "telecom":
    exact_quality_rows = []
    for field in metric_fields:
        profile = metadata_column_profile(panel_parquet, field)
        exact_quality_rows.append({
            "field": field,
            "scope": "complete telecom panel",
            **profile,
        })
    variable_quality = pd.DataFrame(exact_quality_rows)
    per_series_rows = []
    for entity_id, frame in analysis_sample.groupby(entity_field):
        for field in metric_fields:
            values = pd.to_numeric(frame[field], errors="coerce")
            observed = values.dropna()
            per_series_rows.append({
                "series_id": entity_id,
                "field": field,
                "sample_rows": len(values),
                "missing_fraction": values.isna().mean(),
                "all_missing": observed.empty,
                "frozen_non_missing_values": (
                    len(observed) > 0 and observed.nunique() == 1
                ),
                "scope": "bounded row sample by ONT",
            })
    series_variable_quality = pd.DataFrame(per_series_rows)
else:
    variable_quality = (
        metadata_variable_quality.groupby(
            ["source_kind", "field"], as_index=False
        )
        .agg(
            instances=("instance_id", "size"),
            total_rows=("rows", "sum"),
            null_count=("null_count", "sum"),
            completely_missing_instances=("all_missing", "sum"),
            frozen_instances=("frozen_non_missing_values", "sum"),
            metadata_statistics_complete=(
                "metadata_statistics_complete", "all"
            ),
        )
    )
    variable_quality["missing_observation_fraction"] = (
        variable_quality["null_count"] / variable_quality["total_rows"]
    )
    series_variable_quality = metadata_variable_quality[
        [
            "instance_id", "event_code", "source_kind", "field",
            "rows", "missing_fraction", "all_missing",
            "frozen_non_missing_values", "metadata_statistics_complete",
        ]
    ].copy()

write_csv(variable_quality, "variable_quality.csv")
write_csv(series_variable_quality, "series_variable_quality.csv")
display(variable_quality)

if EDA_SECTOR == "telecom":
    plot_quality = variable_quality.set_index("field")
    missing_values = plot_quality["missing_fraction"]
    frozen_values = (
        series_variable_quality.groupby("field")[
            "frozen_non_missing_values"
        ].mean()
    )
else:
    plot_quality = variable_quality.loc[
        variable_quality["source_kind"].eq("real")
    ].set_index("field")
    missing_values = plot_quality["missing_observation_fraction"]
    frozen_values = (
        plot_quality["frozen_instances"] / plot_quality["instances"]
    )

figure, axes = plt.subplots(1, 2, figsize=(15, max(5, len(metric_fields) * 0.32)))
axes[0].barh(missing_values.index, missing_values.values, color="#4C78A8")
axes[0].set_title("Missing observations")
axes[0].set_xlabel("fraction")
axes[0].grid(axis="x", alpha=0.25)
axes[1].barh(frozen_values.index, frozen_values.values, color="#F58518")
axes[1].set_title("Series or instances with one observed value")
axes[1].set_xlabel("fraction")
axes[1].grid(axis="x", alpha=0.25)
save_figure(figure, "01_missing_and_frozen.png")


## 6. Descriptive statistics and empirical distributions


In [ ]:
def descriptive_statistics(frame, fields):
    rows = []
    for field in fields:
        values = pd.to_numeric(frame[field], errors="coerce")
        finite = values[np.isfinite(values)].dropna()
        quantiles = finite.quantile(
            [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
        )
        iqr = quantiles.get(0.75, np.nan) - quantiles.get(0.25, np.nan)
        bowley = (
            (
                quantiles.get(0.75)
                + quantiles.get(0.25)
                - 2 * quantiles.get(0.50)
            ) / iqr
            if iqr > 0 else np.nan
        )
        median = finite.median() if len(finite) else np.nan
        rows.append({
            "field": field,
            "scope": sample_scope,
            "sample_rows": len(values),
            "valid_count": len(finite),
            "missing_fraction": values.isna().mean(),
            "unique_values": finite.nunique(),
            "zero_fraction_of_valid": finite.eq(0).mean() if len(finite) else np.nan,
            "mean": finite.mean() if len(finite) else np.nan,
            "standard_deviation": finite.std() if len(finite) else np.nan,
            "minimum": finite.min() if len(finite) else np.nan,
            "p01": quantiles.get(0.01, np.nan),
            "p05": quantiles.get(0.05, np.nan),
            "p25": quantiles.get(0.25, np.nan),
            "median": median,
            "p75": quantiles.get(0.75, np.nan),
            "p95": quantiles.get(0.95, np.nan),
            "p99": quantiles.get(0.99, np.nan),
            "maximum": finite.max() if len(finite) else np.nan,
            "iqr": iqr,
            "median_absolute_deviation": (
                np.median(np.abs(finite - median)) if len(finite) else np.nan
            ),
            "moment_skew": finite.skew() if len(finite) > 2 else np.nan,
            "bowley_skew": bowley,
            "minimum_value_fraction": (
                finite.eq(finite.min()).mean() if len(finite) else np.nan
            ),
            "maximum_value_fraction": (
                finite.eq(finite.max()).mean() if len(finite) else np.nan
            ),
        })
    return pd.DataFrame(rows)


descriptive_summary = descriptive_statistics(analysis_sample, metric_fields)
write_csv(descriptive_summary, "descriptive_statistics.csv")
display(descriptive_summary)


In [ ]:
columns = 3
rows = math.ceil(len(focus_metrics) / columns)
figure, axes = plt.subplots(
    rows, columns, figsize=(15, 3.7 * rows), squeeze=False
)
for axis, field in zip(axes.flat, focus_metrics):
    values = pd.to_numeric(
        analysis_sample[field], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan).dropna()
    if values.nunique() <= 1:
        axis.text(0.5, 0.5, "constant or unavailable", ha="center")
    else:
        lower, upper = values.quantile([0.005, 0.995])
        axis.hist(
            values.clip(lower, upper),
            bins=60,
            color="#4C78A8",
            alpha=0.85,
        )
    axis.set_title(field)
    axis.set_xlabel("value; display limited to sample 0.5%–99.5%")
    axis.set_ylabel("sample rows")
    axis.grid(alpha=0.2)
for axis in axes.flat[len(focus_metrics):]:
    axis.axis("off")
save_figure(figure, "02_feature_distributions.png")


## 7. Complete selected time series


In [ ]:
if EDA_SECTOR == "telecom":
    selected_entities = (
        pd.Series(topology["ont_id"].astype(str).unique())
        .sort_values(key=lambda values: values.map(
            lambda value: hash_fraction(value, "complete_series")
        ))
        .head(SERIES_COUNT)
        .tolist()
    )
    series_parts = []
    read_columns = [
        timestamp_field, entity_field,
        *metric_fields,
        *[field for field in ["gt_state"] if field in label_fields],
    ]
    for batch in panel_parquet.iter_batches(
        batch_size=100000, columns=read_columns
    ):
        frame = batch.to_pandas()
        selected = frame.loc[
            frame[entity_field].astype(str).isin(selected_entities)
        ].copy()
        if len(selected):
            series_parts.append(selected)
    series_data = pd.concat(series_parts, ignore_index=True)
    series_data["series_id"] = series_data[entity_field].astype(str)
    complete_series_manifest = pd.DataFrame({
        "series_id": selected_entities,
        "entity_id": selected_entities,
    })
else:
    preferred_event_order = [0, 1, 2, 5, 6, 7, 8, 9, 3, 4]
    selected_records = []
    for event_code in preferred_event_order:
        candidates = (
            real_inventory.loc[real_inventory["event_code"].eq(event_code)]
            .assign(order=lambda frame: frame["instance_id"].map(
                lambda value: hash_fraction(value, "complete_series")
            ))
            .sort_values("order")
        )
        if len(candidates):
            selected_records.append(candidates.iloc[0])
        if len(selected_records) == SERIES_COUNT:
            break
    complete_series_files = pd.DataFrame(selected_records)
    series_parts = []
    for record in complete_series_files.itertuples(index=False):
        frame = read_threew(
            record.path,
            [timestamp_field, *metric_fields, *label_fields],
        )
        frame[entity_field] = record.entity_id
        frame["series_id"] = record.instance_id
        frame["event_code"] = record.event_code
        series_parts.append(frame)
    series_data = pd.concat(series_parts, ignore_index=True)
    complete_series_manifest = complete_series_files[
        [
            "instance_id", "event_code", "source_kind",
            "entity_id", "rows", "start_ts", "end_ts",
        ]
    ].rename(columns={"instance_id": "series_id"})

series_data[timestamp_field] = pd.to_datetime(
    series_data[timestamp_field], utc=True, errors="coerce"
)
series_data = series_data.sort_values(
    ["series_id", timestamp_field]
).reset_index(drop=True)
write_csv(complete_series_manifest, "complete_series_manifest.csv")
display(complete_series_manifest)
print(f"Complete-series rows loaded: {len(series_data):,}")
display(
    series_data[
        [timestamp_field, entity_field, "series_id", *focus_metrics,
         *[field for field in label_fields if field in series_data]]
    ].head(10)
)


In [ ]:
series_ids = list(series_data["series_id"].drop_duplicates())
plot_fields = [*focus_metrics]
if EDA_SECTOR == "petrobras_3w":
    plot_fields.append("class")
elif "gt_state" in series_data:
    plot_fields.append("gt_state")

figure, axes = plt.subplots(
    len(series_ids),
    len(plot_fields),
    figsize=(4.0 * len(plot_fields), 2.7 * len(series_ids)),
    squeeze=False,
)
for row_index, series_id in enumerate(series_ids):
    frame = series_data.loc[
        series_data["series_id"].eq(series_id)
    ].sort_values(timestamp_field)
    if len(frame) > SERIES_MAX_PLOT_POINTS:
        positions = np.linspace(
            0, len(frame) - 1, SERIES_MAX_PLOT_POINTS, dtype=int
        )
        displayed = frame.iloc[positions]
    else:
        displayed = frame
    for column_index, field in enumerate(plot_fields):
        axis = axes[row_index, column_index]
        values = pd.to_numeric(displayed[field], errors="coerce")
        if values.notna().any():
            axis.plot(displayed[timestamp_field], values, linewidth=0.75)
            axis.tick_params(axis="x", rotation=30)
        else:
            axis.text(
                0.5, 0.5, "no observed values",
                ha="center", va="center", transform=axis.transAxes,
            )
            axis.set_xticks([])
        axis.set_title(f"{series_id}\n{field}", fontsize=8)
        axis.grid(alpha=0.2)
save_figure(figure, "03_complete_time_series.png")


## 8. Time-series sampling, autocorrelation, trend and spectrum


In [ ]:
def longest_contiguous_observed(frame, field, cadence):
    values = pd.to_numeric(frame[field], errors="coerce")
    timestamps = frame[timestamp_field]
    breaks = (
        values.isna()
        | timestamps.diff().ne(cadence)
        | values.shift().isna()
    )
    groups = breaks.cumsum()
    valid_frame = pd.DataFrame({
        "timestamp": timestamps,
        "value": values,
        "group": groups,
    }).dropna(subset=["value", "timestamp"])
    if valid_frame.empty:
        return valid_frame
    largest_group = valid_frame.groupby("group").size().idxmax()
    return valid_frame.loc[
        valid_frame["group"].eq(largest_group),
        ["timestamp", "value"],
    ]


temporal_rows = []
for series_id, frame in series_data.groupby("series_id", sort=True):
    frame = frame.sort_values(timestamp_field)
    timestamps = frame[timestamp_field].dropna()
    unique_ts = timestamps.drop_duplicates()
    differences = unique_ts.diff().dropna()
    positive = differences[differences.gt(pd.Timedelta(0))]
    cadence = positive.median() if len(positive) else pd.NaT
    duration = unique_ts.max() - unique_ts.min() if len(unique_ts) else pd.NaT
    expected = (
        int(duration / cadence) + 1
        if pd.notna(cadence) and cadence > pd.Timedelta(0)
        else np.nan
    )
    temporal_rows.append({
        "series_id": series_id,
        "rows": len(frame),
        "start_ts": unique_ts.min(),
        "end_ts": unique_ts.max(),
        "duration_hours": (
            duration.total_seconds() / 3600 if pd.notna(duration) else np.nan
        ),
        "duplicate_timestamps": int(timestamps.duplicated().sum()),
        "median_cadence_seconds": (
            cadence.total_seconds() if pd.notna(cadence) else np.nan
        ),
        "expected_regular_grid_rows": expected,
        "observed_grid_fraction": (
            len(unique_ts) / expected if expected and expected > 0 else np.nan
        ),
        "gaps_over_1_5_cadences": (
            int(positive.gt(cadence * 1.5).sum())
            if pd.notna(cadence) else np.nan
        ),
        "p99_gap_seconds": (
            positive.quantile(0.99).total_seconds()
            if len(positive) else np.nan
        ),
    })

temporal_summary = pd.DataFrame(temporal_rows)
write_csv(temporal_summary, "temporal_summary.csv")
display(temporal_summary)


In [ ]:
candidate_lags = (
    [1, 4, 24, 96, 672]
    if EDA_SECTOR == "telecom"
    else [1, 5, 60, 300, 3600]
)
acf_rows, trend_rows, spectrum_rows, stationarity_rows = [], [], [], []

for series_id, frame in series_data.groupby("series_id", sort=True):
    frame = frame.sort_values(timestamp_field).reset_index(drop=True)
    timestamps = frame[timestamp_field]
    positive = timestamps.drop_duplicates().diff().dropna()
    positive = positive[positive.gt(pd.Timedelta(0))]
    cadence = positive.median() if len(positive) else pd.NaT
    if pd.isna(cadence):
        continue
    times_ns = timestamps.astype("int64").to_numpy()
    cadence_ns = int(cadence.value)
    for field in focus_metrics:
        values = pd.to_numeric(frame[field], errors="coerce").to_numpy(dtype=float)
        for lag in candidate_lags:
            if len(values) <= lag:
                continue
            valid = (
                np.isfinite(values[:-lag])
                & np.isfinite(values[lag:])
                & ((times_ns[lag:] - times_ns[:-lag]) == lag * cadence_ns)
            )
            pairs = int(valid.sum())
            correlation = (
                np.corrcoef(values[:-lag][valid], values[lag:][valid])[0, 1]
                if pairs >= 20
                and np.std(values[:-lag][valid]) > 0
                and np.std(values[lag:][valid]) > 0
                else np.nan
            )
            acf_rows.append({
                "series_id": series_id,
                "field": field,
                "lag_observations": lag,
                "lag_seconds": lag * cadence.total_seconds(),
                "observed_pairs": pairs,
                "pearson_autocorrelation": correlation,
                "missing_values_imputed": False,
            })

        observed = np.flatnonzero(np.isfinite(values))
        if len(observed) >= 20:
            selected = observed[
                np.linspace(0, len(observed) - 1, min(500, len(observed)), dtype=int)
            ]
            elapsed_days = (
                times_ns[selected] - times_ns[selected][0]
            ) / (86400 * 1e9)
            if np.ptp(elapsed_days) > 0 and np.std(values[selected]) > 0:
                slope, intercept, low, high = stats.theilslopes(
                    values[selected], elapsed_days
                )
                trend_rows.append({
                    "series_id": series_id,
                    "field": field,
                    "observations_used": len(selected),
                    "theil_sen_slope_per_day": slope,
                    "slope_ci_low": low,
                    "slope_ci_high": high,
                })

        contiguous = longest_contiguous_observed(frame, field, cadence)
        if len(contiguous) >= 100:
            test_values = contiguous["value"].iloc[:5000].to_numpy(dtype=float)
            if np.std(test_values) > 0:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    try:
                        adf_result = adfuller(
                            test_values, autolag="AIC", maxlag=min(20, len(test_values) // 10)
                        )
                        adf_stat, adf_p, adf_lags = adf_result[:3]
                    except Exception:
                        adf_stat, adf_p, adf_lags = np.nan, np.nan, np.nan
                    try:
                        kpss_result = kpss(
                            test_values, regression="c", nlags="auto"
                        )
                        kpss_stat, kpss_p, kpss_lags = kpss_result[:3]
                    except Exception:
                        kpss_stat, kpss_p, kpss_lags = np.nan, np.nan, np.nan
                stationarity_rows.append({
                    "series_id": series_id,
                    "field": field,
                    "contiguous_observations_used": len(test_values),
                    "adf_statistic": adf_stat,
                    "adf_p_value_unit_root_null": adf_p,
                    "adf_lags": adf_lags,
                    "kpss_statistic": kpss_stat,
                    "kpss_p_value_stationary_null": kpss_p,
                    "kpss_lags": kpss_lags,
                })
                detrended = signal.detrend(test_values)
                frequencies, power = signal.periodogram(
                    detrended,
                    fs=1 / cadence.total_seconds(),
                )
                valid_frequency = frequencies > 0
                if valid_frequency.any():
                    local_frequencies = frequencies[valid_frequency]
                    local_power = power[valid_frequency]
                    peak = int(np.argmax(local_power))
                    spectrum_rows.append({
                        "series_id": series_id,
                        "field": field,
                        "contiguous_observations_used": len(test_values),
                        "peak_frequency_hz": local_frequencies[peak],
                        "peak_period_seconds": 1 / local_frequencies[peak],
                        "peak_power": local_power[peak],
                        "detrending": "linear",
                        "missing_values_imputed": False,
                    })

autocorrelation_summary = pd.DataFrame(acf_rows)
trend_summary = pd.DataFrame(trend_rows)
stationarity_summary = pd.DataFrame(stationarity_rows)
spectrum_summary = pd.DataFrame(spectrum_rows)
write_csv(autocorrelation_summary, "autocorrelation_summary.csv")
write_csv(trend_summary, "trend_summary.csv")
write_csv(stationarity_summary, "stationarity_tests.csv")
write_csv(spectrum_summary, "spectrum_summary.csv")
display(autocorrelation_summary.head(30))
display(trend_summary.head(30))
display(stationarity_summary.head(30))
display(spectrum_summary.head(30))


ADF reports a p-value for a unit-root null hypothesis. KPSS reports a p-value for a
stationary null hypothesis. Both are calculated on the longest contiguous observed
segment, limited to 5,000 observations. No missing value is filled.


## 9. Feature dependence


In [ ]:
correlation_fields = [
    field for field in focus_metrics
    if analysis_sample[field].notna().sum() >= 50
    and analysis_sample[field].nunique(dropna=True) > 1
]
correlation_sample = analysis_sample[correlation_fields]
if len(correlation_sample) > 50000:
    correlation_sample = correlation_sample.sample(
        50000, random_state=RANDOM_SEED
    )
pooled_spearman = correlation_sample.corr(method="spearman")

change_parts = []
for _, frame in series_data.groupby("series_id"):
    change_parts.append(
        frame.sort_values(timestamp_field)[correlation_fields]
        .apply(pd.to_numeric, errors="coerce")
        .diff()
    )
first_difference_spearman = pd.concat(change_parts).corr(method="spearman")

write_csv(pooled_spearman, "correlation_pooled_spearman.csv", index=True)
write_csv(
    first_difference_spearman,
    "correlation_first_difference_spearman.csv",
    index=True,
)

figure, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for axis, matrix, title in [
    (axes[0], pooled_spearman, "sampled levels"),
    (axes[1], first_difference_spearman, "complete-series first differences"),
]:
    image = axis.imshow(matrix, vmin=-1, vmax=1, cmap="coolwarm")
    axis.set_xticks(range(len(matrix)))
    axis.set_yticks(range(len(matrix)))
    axis.set_xticklabels(matrix.columns, rotation=90)
    axis.set_yticklabels(matrix.index)
    axis.set_title(title)
figure.colorbar(image, ax=axes.ravel().tolist(), fraction=0.025)
save_figure(figure, "04_feature_dependence.png")


## 10. Labels, periods and class balance


In [ ]:
def label_text(value):
    return "<NA>" if pd.isna(value) else str(int(value))


label_count_rows = []
if EDA_SECTOR == "petrobras_3w" and FULL_LABEL_SCAN:
    for position, record in enumerate(file_inventory.itertuples(index=False), start=1):
        labels = read_threew(record.path, ["class", "state"])
        for field in ["class", "state"]:
            counts = labels[field].value_counts(dropna=False)
            for value, count in counts.items():
                label_count_rows.append({
                    "source_kind": record.source_kind,
                    "event_code": record.event_code,
                    "label_field": field,
                    "label": label_text(value),
                    "rows": int(count),
                })
        if position % 400 == 0:
            print(f"Scanned labels for {position:,}/{len(file_inventory):,} instances")
    label_distribution = (
        pd.DataFrame(label_count_rows)
        .groupby(
            ["source_kind", "event_code", "label_field", "label"],
            as_index=False,
        )["rows"].sum()
    )
elif EDA_SECTOR == "petrobras_3w":
    for field in ["class", "state"]:
        counts = analysis_sample[field].value_counts(dropna=False)
        for value, count in counts.items():
            label_count_rows.append({
                "source_kind": "real_sample",
                "event_code": None,
                "label_field": field,
                "label": label_text(value),
                "rows": int(count),
            })
    label_distribution = pd.DataFrame(label_count_rows)
else:
    for field in ["gt_state", "gt_fault_type"]:
        if field not in analysis_sample:
            continue
        counts = analysis_sample[field].astype("string").value_counts(dropna=False)
        for value, count in counts.items():
            label_count_rows.append({
                "label_field": field,
                "label": str(value),
                "rows": int(count),
                "sample_fraction": count / len(analysis_sample),
            })
    label_distribution = pd.DataFrame(label_count_rows)

write_csv(label_distribution, "label_distribution.csv")
display(label_distribution)


In [ ]:
period_rows = []
label_field = "class" if EDA_SECTOR == "petrobras_3w" else "gt_state"
if label_field in series_data:
    for series_id, frame in series_data.groupby("series_id", sort=True):
        frame = frame.sort_values(timestamp_field).reset_index(drop=True)
        labels = frame[label_field]
        times = frame[timestamp_field]
        positive = times.diff().dropna()
        positive = positive[positive.gt(pd.Timedelta(0))]
        cadence = positive.median() if len(positive) else pd.NaT
        run_id = labels.ne(labels.shift()) | labels.isna() | labels.shift().isna()
        for _, run in frame.assign(_run=run_id.cumsum()).groupby("_run"):
            value = run[label_field].iloc[0]
            period_rows.append({
                "series_id": series_id,
                "label_field": label_field,
                "label": "<NA>" if pd.isna(value) else str(value),
                "observations": len(run),
                "start_ts": run[timestamp_field].iloc[0],
                "end_ts": run[timestamp_field].iloc[-1],
                "duration_seconds": (
                    len(run) * cadence.total_seconds()
                    if pd.notna(cadence) else np.nan
                ),
            })

label_periods = pd.DataFrame(period_rows)
write_csv(label_periods, "label_periods_selected_series.csv")
display(label_periods)


## 11. Petrobras 3W anomaly-benchmark eligibility

This section is produced only for `petrobras_3w`.

The strict 2019 anomaly-detection benchmark uses:

- real instances only;
- event types 1, 2, 5, 6, 7 and 8;
- instances containing a continuous normal period of at least 20 minutes;
- normal observations as the negative class;
- transient and steady undesirable-event observations as the positive class;
- an initial part of the eligible normal period for one-class training;
- remaining normal observations and all positive observations for testing;
- one independent round per eligible instance;
- precision, recall and F1, with mean F1 as the main reported measure.

Event type 9 and the 27-variable/state-label structure were introduced after the
2019 paper. They are reported separately and are not silently added to the strict
benchmark.

Reference: Vargas et al., *A realistic and public dataset with rare undesirable real
events in oil wells*, Journal of Petroleum Science and Engineering 181 (2019),
106223, https://doi.org/10.1016/j.petrol.2019.106223.


In [ ]:
if EDA_SECTOR == "petrobras_3w":
    strict_event_types = {1, 2, 5, 6, 7, 8}
    v2_extension_types = strict_event_types | {9}
    benchmark_rows = []
    real_events = real_inventory.loc[
        real_inventory["event_code"].ne(0)
    ].sort_values(["event_code", "start_ts"])
    for position, record in enumerate(real_events.itertuples(index=False), start=1):
        frame = read_threew(record.path, ["timestamp", "class"])
        labels = pd.to_numeric(frame["class"], errors="coerce")
        timestamps = pd.to_datetime(frame["timestamp"], utc=True)
        differences = timestamps.diff().dropna()
        positive_differences = differences[differences.gt(pd.Timedelta(0))]
        cadence = (
            positive_differences.median()
            if len(positive_differences) else pd.NaT
        )
        is_normal = labels.eq(0).fillna(False).to_numpy(dtype=bool)
        normal_runs = []
        run_start = None
        for index, value in enumerate(is_normal):
            if value and run_start is None:
                run_start = index
            if run_start is not None and (not value or index == len(is_normal) - 1):
                run_end = index + 1 if value and index == len(is_normal) - 1 else index
                normal_runs.append((run_start, run_end - run_start))
                run_start = None
        first_normal_observations = normal_runs[0][1] if normal_runs else 0
        longest_normal_observations = (
            max(length for _, length in normal_runs) if normal_runs else 0
        )
        first_normal_seconds = (
            first_normal_observations * cadence.total_seconds()
            if pd.notna(cadence) else np.nan
        )
        longest_normal_seconds = (
            longest_normal_observations * cadence.total_seconds()
            if pd.notna(cadence) else np.nan
        )
        is_transient = labels.eq(100 + record.event_code)
        is_steady = labels.eq(record.event_code)
        positive_rows = int((is_transient | is_steady).sum())
        benchmark_rows.append({
            "instance_id": record.instance_id,
            "entity_id": record.entity_id,
            "event_code": record.event_code,
            "rows": len(frame),
            "cadence_seconds": (
                cadence.total_seconds() if pd.notna(cadence) else np.nan
            ),
            "normal_period_count": len(normal_runs),
            "first_normal_period_observations": first_normal_observations,
            "first_normal_period_seconds": first_normal_seconds,
            "longest_normal_period_observations": longest_normal_observations,
            "longest_normal_period_seconds": longest_normal_seconds,
            "transient_observations": int(is_transient.sum()),
            "steady_event_observations": int(is_steady.sum()),
            "positive_observations": positive_rows,
            "unknown_class_observations": int(labels.isna().sum()),
            "eligible_2019_strict": (
                record.event_code in strict_event_types
                and longest_normal_seconds >= 20 * 60
                and positive_rows > 0
            ),
            "eligible_v2_event9_extension": (
                record.event_code in v2_extension_types
                and longest_normal_seconds >= 20 * 60
                and positive_rows > 0
            ),
        })
        if position % 50 == 0:
            print(f"Checked {position:,}/{len(real_events):,} real event instances")
    benchmark_eligibility = pd.DataFrame(benchmark_rows)
    benchmark_summary = (
        benchmark_eligibility.groupby("event_code", as_index=False)
        .agg(
            real_event_instances=("instance_id", "size"),
            strict_eligible_instances=("eligible_2019_strict", "sum"),
            v2_extension_eligible_instances=(
                "eligible_v2_event9_extension", "sum"
            ),
            median_longest_normal_minutes=(
                "longest_normal_period_seconds",
                lambda values: values.median() / 60,
            ),
            minimum_longest_normal_minutes=(
                "longest_normal_period_seconds",
                lambda values: values.min() / 60,
            ),
            total_positive_observations=("positive_observations", "sum"),
        )
    )
    benchmark_protocol = pd.DataFrame([
        {"item": "instance source", "2019 benchmark rule": "real only"},
        {"item": "event types", "2019 benchmark rule": "1, 2, 5, 6, 7, 8"},
        {"item": "minimum continuous normal period", "2019 benchmark rule": "20 minutes"},
        {"item": "negative class", "2019 benchmark rule": "class 0"},
        {"item": "positive class", "2019 benchmark rule": "transient and steady event labels"},
        {"item": "training data", "2019 benchmark rule": "initial part of an eligible normal period only"},
        {"item": "test data", "2019 benchmark rule": "remaining normal and all positive samples"},
        {"item": "learning formulation", "2019 benchmark rule": "one-class; one round per instance"},
        {"item": "reported metrics", "2019 benchmark rule": "precision, recall, F1; mean and standard deviation"},
        {"item": "main metric", "2019 benchmark rule": "mean F1"},
    ])
    write_csv(benchmark_eligibility, "threew_anomaly_benchmark_eligibility.csv")
    write_csv(benchmark_summary, "threew_anomaly_benchmark_summary.csv")
    write_csv(benchmark_protocol, "threew_anomaly_benchmark_protocol.csv")
    display(benchmark_protocol)
    display(benchmark_summary)
    display(
        benchmark_eligibility.loc[
            benchmark_eligibility["eligible_v2_event9_extension"]
        ].head(20)
    )
else:
    benchmark_eligibility = pd.DataFrame()
    benchmark_summary = pd.DataFrame()
    print("3W benchmark eligibility is not applicable to the telecom source.")


## 12. Output inventory


In [ ]:
report = {
    "eda_version": "native_eda_simple_v3",
    "sector": EDA_SECTOR,
    "source_root": str(SOURCE_ROOT),
    "output_root": str(OUTPUT),
    "measurement_fields": len(metric_fields),
    "analysis_sample_rows": len(analysis_sample),
    "complete_series": int(series_data["series_id"].nunique()),
    "large_native_data_copied": False,
    "detector_fitted": False,
    "runtime": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "pyarrow": pyarrow.__version__,
        "scipy": scipy.__version__,
    },
    "artifacts": sorted(
        str(path.relative_to(OUTPUT))
        for path in OUTPUT.rglob("*")
        if path.is_file()
    ),
}
write_json(report, "eda_report.json")
display(pd.Series(report, name="value").to_frame())
print("EDA outputs:", OUTPUT)


Notebook 01 uses the telecom findings to define and validate the Telecom Pack and
canonical translation. Notebook 02 uses the 3W findings to challenge the neutral
contract. Model fitting and benchmark train/test construction belong in Notebook 03,
not in this EDA notebook.
